# 16 - Review round 2: fuzzy parameters and competing optimizers

**Reviewer comments answered.**
- *Comment 9:* "How did you compute the parameters of the fuzzy logic part of the paper? It seems to me that certain
  heuristics are present there." The target orness (0.30) and the gap penalty (0.50) were fixed in advance. This
  notebook measures what happens when they are changed.
- *Comment 11:* "Ensure a fair comparison versus competing approaches ... similar complexity and design using the same
  design approach." Hunger Games Search is compared with particle swarm optimization, a genetic algorithm and random
  search on the same objective, the same grid, the same splits and the same number of distinct model evaluations.

**Design.** For every task (T1, T1b, T2, T3) and split (seeds 42, 7, 123):

| block | runs | what changes |
|---|---|---|
| orness sweep | HGS with target orness 0.10, 0.20, 0.30, 0.50, 1.00 (gap penalty 0.50) | pessimism of both OWA operators; 1.00 is the arithmetic mean |
| penalty sweep | HGS with gap penalty 0, 0.25, 0.50, 1.00, 2.00 (orness 0.30) | weight of the train-CV gap |
| optimizers | HGS, random search, PSO, GA on the paper's objective (0.30, 0.50) | search method; budget = the number of distinct models HGS evaluated |

**How it stays cheap.** Each model evaluation stores the raw criteria (F1, recall and ROC area of every fold and of the
training split), not the fitness. The fitness of any (orness, penalty) pair is then recomputed from the stored
criteria, so a model fitted for one setting or optimizer is never fitted again for another. The test split is scored
once per fitted model and stored, but it never enters a fitness value.

**Check built in.** HGS at (0.30, 0.50) must select the same model as the paper's stored run in `results_all.csv`;
the report says whether it did.

**Runtime.** Unknown in advance, because it depends on how much the searches overlap. Expect roughly 40 to 120 new
models per task and split, at 20 to 90 seconds each: several hours per task on a CPU runtime. Everything is resumable.
To split the work, open the notebook in several sessions and give each one task in `SENS_TASKS`; each task writes its
own result file.

**Outputs:** `results_sensitivity_<task>.csv`, `sensitivity_report.txt`, `fig_sensitivity.png`, predictions named
`sens__<task>_s<seed>_<run>`.

## Shared setup (identical in every job notebook)

Run these cells in order. The embeddings are loaded from the cache written by the first run, so a **CPU runtime is enough** unless the cache is missing.

In [ ]:
!pip -q install transformers scikit-learn scipy
import os
try:                                   # Google Colab: data and results on Google Drive
    from google.colab import drive
    drive.mount('/content/drive'); IN_COLAB = True
except ImportError:                    # local run from the notebooks/ folder
    IN_COLAB = False
DATA_DIR = os.environ.get("LINKEDIN_DATA_DIR",
    "/content/drive/MyDrive/fake_profile_paper" if IN_COLAB else os.path.abspath("../data"))
OUT_DIR = os.environ.get("LINKEDIN_OUT_DIR",
    "/content/drive/MyDrive/fake_profile_paper/linkedin_repro" if IN_COLAB else os.path.abspath("../results/run"))
print("data:", DATA_DIR, "| results:", OUT_DIR)

import os, ast, time, logging
import numpy as np, pandas as pd, torch, joblib
from transformers import RobertaTokenizer, RobertaModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
RANDOM_STATE = 42
os.makedirs('embeddings_output', exist_ok=True)
os.makedirs('saved_models', exist_ok=True)

# ==== CELL ====
import json, math, random, itertools, warnings
from sklearn.metrics import average_precision_score, matthews_corrcoef, balanced_accuracy_score
warnings.filterwarnings("ignore")
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- experiment switches ----------------
SEEDS            = [42, 7, 123]      # 3 seeds ~ comfortable in one session; add 2024, 99 for 5
RUN_TASKS        = ["T1", "T1b", "T2"] # add "T3" if wanted
RUN_RANDOM       = True              # matched-budget random search
RUN_GRID         = True              # exhaustive 336-model search (first seed only; slow)
RUN_ABLATIONS    = True              # tag correction off / Full Name excluded (first seed)
FAITHFUL_REPLAY  = False             # already verified in the first run
HGS_KW = dict(pop_size=12, max_iter=18, early_stop_patience=4)   # as in the original run

In [ ]:
def load_raw():
    csv = os.path.join(DATA_DIR, "LinkedIn_Dataset.csv")
    pcl = os.path.join(DATA_DIR, "LinkedIn_Dataset.pcl")
    if os.path.exists(csv):
        df = pd.read_csv(csv); col = "test" if "test" in df.columns else "Label"
    elif os.path.exists(pcl):
        df = pd.read_pickle(pcl); col = "Label" if "Label" in df.columns else "test"
    else:
        raise FileNotFoundError("put LinkedIn_Dataset.csv or .pcl in " + DATA_DIR)
    code = pd.to_numeric(df[col], errors="coerce").astype(int).values
    return df.reset_index(drop=True), code

import hashlib
OFFICIAL_SHA256 = "6d67c8015fceee06226491a8b0683bfc01e220416fed99e7f2d91aca27ba2e1a"   # LinkedIn_Dataset.pcl, github.com/navid-aub/LinkedIn-Dataset
_p = os.path.join(DATA_DIR, "LinkedIn_Dataset.pcl")
if os.path.exists(os.path.join(DATA_DIR, "LinkedIn_Dataset.csv")):
    print("note: LinkedIn_Dataset.csv is present and is used instead of the official .pcl file")
elif os.path.exists(_p):
    _h = hashlib.sha256(open(_p, "rb").read()).hexdigest()
    print("dataset checksum:", "matches the official release" if _h == OFFICIAL_SHA256
          else "WARNING - differs from the official release (%s)" % _h)

dataset, RAW_CODE = load_raw()
counts = pd.Series(RAW_CODE).value_counts().sort_index().to_dict()
print("rows:", len(dataset), "| raw code counts:", counts)
assert counts == {0: 1800, 1: 600, 10: 600, 11: 600}, "counts differ from the README - check the file"
print("OK: matches the README (0=LLP, 1=FLP, 10/11=ChatGPT-generated)")

In [ ]:
TASKS = {
  "T1":  dict(neg=[0],    pos=[1],        balance_neg=None, name="legitimate vs manual fake (Gulati protocol)"),
  "T1b": dict(neg=[0],    pos=[1],        balance_neg=600,  name="legitimate vs manual fake (Ayoobi, balanced)"),
  "T2":  dict(neg=[0, 1], pos=[10, 11],   balance_neg=None, name="human-made vs ChatGPT-generated (original headline)"),
  "T3":  dict(neg=[0],    pos=[1, 10, 11],balance_neg=None, name="legitimate vs any fake"),
}
def task_index(task, seed):
    t = TASKS[task]
    neg = np.where(np.isin(RAW_CODE, t["neg"]))[0]
    pos = np.where(np.isin(RAW_CODE, t["pos"]))[0]
    if t["balance_neg"]:
        neg = np.random.default_rng(seed).choice(neg, t["balance_neg"], replace=False)
    idx = np.sort(np.r_[neg, pos])
    y = np.isin(RAW_CODE[idx], t["pos"]).astype(int)
    return idx, y
for k in RUN_TASKS:
    idx, y = task_index(k, 42)
    print("%-4s %-52s n=%-5d positives=%-5d (%.1f%%)" % (k, TASKS[k]["name"], len(y), y.sum(), 100*y.mean()))

In [ ]:
REQUIRED_COLUMNS = [
    'Intro','Full Name','Workplace','Location','Connections','Photo','Followers','About',
    'Experiences','Number of Experiences','Educations','Number of Educations','Licenses',
    'Number of Licenses','Volunteering','Number of Volunteering','Skills','Number of Skills',
    'Recommendations','Number of Recommendations','Projects','Number of Projects',
    'Publications','Number of Publications','Courses','Number of Courses','Honors',
    'Number of Honors','Scores','Number of Scores','Languages','Number of Languages',
    'Organizations','Number of Organizations','Interests','Number of Interests',
    'Activities','Number of Activities','Label']

def get_dtype(col):
    if col == 'Label': return 'Int64'
    if 'Number of' in col: return 'Int64'
    return 'string'

def parse_chatgpt_content(content):
    parts = content.split('#@')
    if len(parts) >= 4:
        return {'Full Name': parts[0].strip(),
                'Workplace': f"{parts[1].strip()} at {parts[2].strip()}",
                'Location':  parts[3].strip(),
                'Connections': '500+', 'Photo': 'No', 'Followers': ''}
    return {k: '' for k in ['Full Name','Workplace','Location','Connections','Photo','Followers']}

def clean_chatgpt_field(field, name):
    try:
        if isinstance(field, str):
            field = ast.literal_eval(field)
        if isinstance(field, dict) and 'chatGPT' in field:
            content = field['chatGPT']
            first = content.split('\n')[0]
            if first[:2] in ('1.','2.','3.','4.','5.'):
                first = first.split('.', 1)[1].strip()
            if name == 'Intro': return parse_chatgpt_content(first)
            if name == 'Full Name': return first.split('#@')[0].strip()
            if name == 'Workplace':
                p = first.split('#@')
                return f"{p[1].strip()} at {p[2].strip()}" if len(p) > 2 else ''
            if name == 'Location':
                p = first.split('#@')
                return p[3].strip() if len(p) > 3 else ''
            return first.replace('#@', ' ')
        if isinstance(field, dict): return field
    except Exception:
        return parse_chatgpt_content('') if name == 'Intro' else ''
    return field if field is not None else ''

def safe_eval(v):
    if isinstance(v, str):
        try: return ast.literal_eval(v)
        except Exception: return v
    return v

def to_float(v):
    try: return float(v)
    except Exception: return 0.0

NUMERICAL_FIELDS = ['Number of Experiences','Number of Educations','Number of Licenses',
                    'Number of Volunteering','Number of Skills','Number of Recommendations',
                    'Number of Projects','Number of Publications','Number of Courses',
                    'Number of Honors','Number of Scores','Number of Languages',
                    'Number of Organizations','Number of Interests','Number of Activities']
NUMERICAL_FIELD_NAMES = NUMERICAL_FIELDS

def clean_dataset(df):
    cleaned = pd.DataFrame(columns=REQUIRED_COLUMNS)
    for c in REQUIRED_COLUMNS:
        cleaned[c] = pd.Series(dtype=get_dtype(c))
    for c in df.columns:
        if c in REQUIRED_COLUMNS and c != 'Label':
            cleaned[c] = df[c]
    if 'Label' in df.columns:
        cleaned['Label'] = pd.to_numeric(df['Label'], errors='coerce').fillna(0).astype(int)
    else:
        # Définition de label : test ≥ 10 → 1 sinon 0 (cohérent avec notebook CatBoost LinkedIn)
        if 'test' in df.columns:
            cleaned['Label'] = df['test'].apply(lambda v: 1 if to_float(v) >= 10 else 0).astype(int)
        else:
            cleaned['Label'] = 0
    for c in ('Intro','Full Name','Workplace','Location'):
        if c in cleaned.columns:
            cleaned[c] = cleaned[c].apply(lambda v: clean_chatgpt_field(v, c))
    # Conditional imputation depuis Intro
    intros = cleaned['Intro'].apply(lambda v: v if isinstance(v, dict) else parse_chatgpt_content(''))
    for k in ('Full Name','Workplace','Location','Connections','Photo','Followers'):
        mask = cleaned[k].astype(str).str.strip().eq('') | cleaned[k].isna()
        cleaned.loc[mask, k] = intros[mask].apply(lambda d: d.get(k, ''))
    cleaned = cleaned.fillna('')
    return cleaned


cleaned_df = clean_dataset(dataset)
print("Cleaned:", cleaned_df.shape)

In [ ]:
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
roberta   = RobertaModel.from_pretrained('roberta-base').to(device).eval()

@torch.no_grad()
def embed_text(text):
    if not text or not text.strip():
        return np.zeros(roberta.config.hidden_size, dtype=np.float32)
    inp = tokenizer(text, return_tensors='pt', padding=True,
                    truncation=True, max_length=512).to(device)
    out = roberta(**inp).last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    return out.astype(np.float32)

def section_text_embedding(content, tag):
    return embed_text(content) - embed_text(tag)


def section_texts(df, include_name=True):
    rows = []
    for i in range(len(df)):
        intro = safe_eval(df['Intro'].iloc[i]); about = safe_eval(df['About'].iloc[i])
        exps  = safe_eval(df['Experiences'].iloc[i])
        if isinstance(intro, dict):
            parts = ([intro.get('Full Name','')] if include_name else []) + \
                    [intro.get('Workplace',''), intro.get('Location','')]
            intro_text = " ".join(str(p) for p in parts)
        else:
            intro_text = str(intro)
        about_text = about.get('About','') if isinstance(about, dict) else str(about)
        exp_text = ""
        if isinstance(exps, dict): exps = exps.values()
        if isinstance(exps, (list, type({}.values()))):
            for e in exps:
                if isinstance(e, dict):
                    exp_text += " ".join(str(e.get(k,'')) for k in
                        ['Role','Workplace','Duration','Workplace Location','Description']) + " "
        rows.append((intro_text, about_text, exp_text))
    return rows

def cached_section_embeddings(include_name):
    path = os.path.join(OUT_DIR, "roberta_sections_name%d.npy" % int(include_name))
    if os.path.exists(path):
        return np.load(path)
    if not torch.cuda.is_available():
        raise RuntimeError('No cached embeddings found at ' + path + ' and no GPU. '
                           'Switch the runtime to GPU T4 for the first run.')
    rows = section_texts(cleaned_df, include_name)
    E = np.zeros((len(rows), 3, roberta.config.hidden_size), dtype=np.float32)
    for i, (a, b, c) in enumerate(rows):
        E[i, 0], E[i, 1], E[i, 2] = embed_text(a), embed_text(b), embed_text(c)
        if (i+1) % 300 == 0: print("  embedded", i+1, "/", len(rows))
    np.save(path, E); return E

TAG_E = np.stack([embed_text("Introduction"), embed_text("About"), embed_text("Experiences")])
SEC_NAME   = cached_section_embeddings(include_name=True)    # original behaviour
try:
    SEC_NONAME = cached_section_embeddings(include_name=False)   # cached by notebook 00
except Exception as e:
    SEC_NONAME = None; print('no-name embeddings not available:', e)

def text_features(sec, tag_correction=True):
    return ((sec - TAG_E[None]) if tag_correction else sec).mean(axis=1)

NUM_X = np.array([[to_float(cleaned_df[f].iloc[i]) for f in NUMERICAL_FIELDS]
                  for i in range(len(cleaned_df))], dtype=np.float32)
print("sections:", SEC_NAME.shape, "| numeric:", NUM_X.shape)

In [ ]:
# =============================================================================
#  MODULE : Hunger Games Search (HGS) + Fuzzy OWA Aggregation Anti-Overfitting
#  ---------------------------------------------------------------------------
#  Référence HGS : Yang Y., Chen H., Heidari A.A., Gandomi A.H. (2021)
#                  "Hunger Games Search", ESWA 177, 114864.
#  Référence OWA  : Yager R.R. (1988) "On Ordered Weighted Averaging Aggregation
#                                      Operators in Multi-Criteria Decision Making"
#
#  STRATÉGIE ANTI-OVERFITTING :
#  ---------------------------------------------------------------------------
#  1)  Le test set n'est JAMAIS vu par le fitness (sacré jusqu'à l'évaluation
#      finale). Fitness = K-fold CV stratifiée sur (X_train, y_train).
#  2)  Agrégation FLOUE (OWA pessimiste) des K scores : on donne plus de poids
#      aux PIRES plis → pénalise la variance entre folds → robustesse.
#  3)  Fuzzy multi-metric : fusion F1 ⊕ AUC ⊕ Recall par OWA pondéré (chaque
#      métrique est min-max normalisée).
#  4)  Pénalité de généralisation : fitness final = OWA_cv - λ · (train - μ_cv)
#      → décourage les modèles qui sur-apprennent (gap train-CV élevé).
#  5)  Early-stopping HGS : si pas d'amélioration pendant `patience` itérations,
#      on arrête → moins d'évaluations, moins de risque de surajustement à un
#      pli particulier.
# =============================================================================
import math, random, time
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.metrics import f1_score, roc_auc_score, recall_score, accuracy_score


# -----------------------------------------------------------------------------
#  1. Fuzzy OWA (Ordered Weighted Average) — Yager 1988
# -----------------------------------------------------------------------------
def owa_weights(n, orness=0.30):
    """
    Génère des poids OWA pour un vecteur de taille n.
        orness ∈ [0,1] :
            • 0.0  → "AND" (min)              très pessimiste
            • 0.5  → moyenne arithmétique
            • 1.0  → "OR"  (max)              très optimiste
        On choisit orness = 0.30 (légèrement pessimiste) : on récompense
        les modèles dont les PIRES folds restent corrects.
    Implémentation : poids exponentiels biaisés vers la fin (queue basse).
    """
    if n <= 1:
        return np.ones(n) / max(n, 1)
    # Distribution exponentielle : alpha contrôle la pente
    # alpha = -ln(orness)  ⇒ pour orness < 0.5, poids concentrés sur les PETITES
    #                       valeurs (premiers éléments du vecteur trié ascending)
    #                       → moyenne pondérée tirée vers le bas = pessimiste.
    alpha = max(-math.log(max(orness, 1e-6)), 1e-6)
    idx = np.arange(n)
    # poids décroissants : w[0] (pire valeur) = max, w[n-1] (meilleure) = min
    w = np.exp(-alpha * idx / max(n - 1, 1))
    return w / w.sum()


def fuzzy_owa(scores, orness=0.30):
    """
    Agrégation OWA d'un vecteur de scores.
    1) Trier les scores en ordre croissant.
    2) Appliquer w_i tel que les premiers (pires) reçoivent plus de poids.
    """
    arr = np.asarray(scores, dtype=float).ravel()
    if arr.size == 0:
        return 0.0
    arr = np.sort(arr)                          # ordre croissant
    w   = owa_weights(arr.size, orness=orness)  # poids biaisés bas
    return float(np.dot(w, arr))


def fuzzy_multi_metric(y_true, y_pred, y_proba=None, orness=0.30):
    """
    Combine F1, Recall et AUC (si y_proba) en un score flou unique via OWA.
    Chaque métrique ∈ [0,1] → comparable.
    """
    binary = len(np.unique(y_true)) == 2
    avg = "binary" if binary else "macro"
    f1  = f1_score(y_true, y_pred, average=avg, zero_division=0)
    rec = recall_score(y_true, y_pred, average=avg, zero_division=0)
    metrics = [f1, rec]
    if y_proba is not None and binary:
        try:
            auc = roc_auc_score(y_true, y_proba)
            metrics.append(auc)
        except Exception:
            pass
    return fuzzy_owa(metrics, orness=orness)


# -----------------------------------------------------------------------------
#  2. Fitness K-fold CV avec fuzzy aggregation + pénalité de gap
# -----------------------------------------------------------------------------
def make_cv_fitness(estimator_factory, X_train, y_train,
                    n_splits=5, seed=42, orness=0.30,
                    gap_penalty=0.50, use_proba=True):
    """
    Renvoie une fonction de fitness `fit(params) -> float` agrégée via OWA flou.

        estimator_factory : callable(params:dict) -> sklearn-compatible model
        n_splits          : nb de folds (5 par défaut)
        orness            : ∈ [0,1], <0.5 = pessimiste, pénalise variance
        gap_penalty       : λ, pénalité du gap (train - μ_cv) ∈ [0,1]
        use_proba         : True si l'estimator expose predict_proba (AUC)

    Le SCORE final renvoyé :
        score = OWA_cv  -  λ · max(0, train_full - mean_cv)
    où :
        OWA_cv     = agrégation floue OWA des K scores fuzzy_multi_metric
        train_full = score fuzzy_multi_metric sur l'ENSEMBLE train re-fit
        gap        = (train_full - mean_cv)  → quantifie l'overfitting
    """
    X_train = np.asarray(X_train) if not hasattr(X_train, "toarray") else X_train
    y_train = np.asarray(y_train)

    def _safe_score(est, X_tr, y_tr, X_va, y_va):
        try:
            est.fit(X_tr, y_tr)
            y_pred = est.predict(X_va)
            y_proba = None
            if use_proba and hasattr(est, "predict_proba"):
                try:
                    y_proba = est.predict_proba(X_va)[:, 1]
                except Exception:
                    y_proba = None
            return fuzzy_multi_metric(y_va, y_pred, y_proba, orness=orness)
        except Exception:
            return 0.0

    def fitness(params):
        # --- K-fold CV stratifiée sur X_train UNIQUEMENT (jamais X_test) ---
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        fold_scores = []
        try:
            est_template = estimator_factory(params)
        except Exception:
            return 0.0

        for tr_idx, va_idx in skf.split(np.zeros(len(y_train)), y_train):
            # sparse-safe indexing
            if hasattr(X_train, "tocsr"):
                Xtr_f, Xva_f = X_train.tocsr()[tr_idx], X_train.tocsr()[va_idx]
            else:
                Xtr_f, Xva_f = X_train[tr_idx], X_train[va_idx]
            ytr_f, yva_f = y_train[tr_idx], y_train[va_idx]
            est = clone(est_template)
            fold_scores.append(_safe_score(est, Xtr_f, ytr_f, Xva_f, yva_f))

        owa_cv  = fuzzy_owa(fold_scores, orness=orness)
        mean_cv = float(np.mean(fold_scores))

        # --- Score train complet pour calcul de gap (overfitting) ---
        est_full = clone(est_template)
        try:
            est_full.fit(X_train, y_train)
            y_pred_tr = est_full.predict(X_train)
            y_proba_tr = None
            if use_proba and hasattr(est_full, "predict_proba"):
                try:
                    y_proba_tr = est_full.predict_proba(X_train)[:, 1]
                except Exception:
                    y_proba_tr = None
            train_full = fuzzy_multi_metric(y_train, y_pred_tr, y_proba_tr, orness=orness)
        except Exception:
            train_full = mean_cv

        gap     = max(0.0, train_full - mean_cv)
        penalty = gap_penalty * gap
        return owa_cv - penalty

    return fitness


# -----------------------------------------------------------------------------
#  3. HGS Optimizer (avec early-stopping pour limiter le risque d'overfit)
# -----------------------------------------------------------------------------
class HGSOptimizer:
    """
    Hunger Games Search — Yang et al. 2021.
    Compatible recherche discrète : search_space = {param: [v1, v2, ...]}.

    Améliorations anti-overfitting :
        • early_stop_patience : arrête HGS si pas d'amélioration sur N itérations
        • Le fitness est typiquement obtenu via make_cv_fitness (fuzzy OWA + gap).
    """
    def __init__(self, search_space, fitness_fn,
                 pop_size=8, max_iter=15,
                 l_h=1e-2, LH=1e4,
                 seed=42, verbose=True,
                 early_stop_patience=4):
        self.space   = search_space
        self.keys    = list(search_space.keys())
        self.dims    = [len(search_space[k]) for k in self.keys]
        self.D       = len(self.keys)
        self.fit     = fitness_fn
        self.N       = pop_size
        self.T       = max_iter
        self.l_h     = l_h
        self.LH      = LH
        self.verbose = verbose
        self.patience = early_stop_patience
        random.seed(seed); np.random.seed(seed)
        self.history = []
        self.best_params_ = None
        self.best_score_  = -np.inf

    def _decode(self, x):
        params = {}
        for i, k in enumerate(self.keys):
            idx = int(min(self.dims[i] - 1, max(0, math.floor(x[i] * self.dims[i]))))
            params[k] = self.space[k][idx]
        return params

    def _evaluate(self, x, cache):
        params = self._decode(x)
        key = tuple(sorted(params.items(), key=lambda kv: kv[0]))
        if key in cache:
            return cache[key], params
        s = self.fit(params)
        cache[key] = s
        return s, params

    def optimize(self):
        X = np.random.rand(self.N, self.D)
        F = np.full(self.N, -np.inf)
        Hunger = np.zeros(self.N)
        cache = {}

        for i in range(self.N):
            F[i], _ = self._evaluate(X[i], cache)

        best_idx = int(np.argmax(F))
        X_best   = X[best_idx].copy()
        F_best   = F[best_idx]
        self.best_params_ = self._decode(X_best)
        self.best_score_  = F_best
        self.history.append(F_best)

        if self.verbose:
            print(f"[HGS] init  best fuzzy-score={F_best:.4f}  params={self.best_params_}")

        no_improve = 0
        for t in range(self.T):
            E_t = 2 * (1 - t / max(1, self.T))
            best_fit  = F.max();  worst_fit = F.min()

            for i in range(self.N):
                if abs(best_fit - worst_fit) < 1e-12:
                    TH = 0
                else:
                    TH = (F[i] - best_fit) / (best_fit - worst_fit + 1e-12) * \
                         random.random() * 2 * (1 - t / max(1, self.T))
                hunger_i = TH if abs(TH) > self.l_h else self.l_h
                Hunger[i] += min(self.LH, abs(hunger_i))

            S_hunger = Hunger.sum() + 1e-12
            W1 = np.zeros(self.N); W2 = np.zeros(self.N)
            for i in range(self.N):
                if random.random() < (Hunger[i] / S_hunger):
                    W1[i] = Hunger[i] * self.N / S_hunger * random.random()
                else:
                    W1[i] = 1.0
                W2[i] = (1 - math.exp(-abs(Hunger[i] - Hunger[best_idx]))) * \
                        random.random() * 2

            improved_iter = False
            for i in range(self.N):
                r1, r2 = random.random(), random.random()
                a = 2 * E_t * r1 - E_t
                R = a
                E = math.copysign(1, random.random() - 0.5) * \
                    (math.exp(-abs(F[i] - F_best) / (abs(F_best) + 1e-12)))

                levy_step = 0.01 * np.random.standard_cauchy(self.D)
                if r1 < self.l_h:
                    X_new = np.random.rand(self.D)
                elif r2 > E:
                    X_new = W1[i] * X_best + R * W2[i] * abs(X_best - X[i])
                else:
                    X_new = W1[i] * X_best - R * W2[i] * abs(X_best - X[i]) + levy_step
                X_new = np.clip(X_new, 0.0, 1.0)

                F_new, params_new = self._evaluate(X_new, cache)
                if F_new > F[i]:
                    X[i] = X_new
                    F[i] = F_new
                    if F_new > F_best:
                        F_best = F_new
                        X_best = X_new.copy()
                        best_idx = i
                        self.best_params_ = params_new
                        self.best_score_  = F_best
                        improved_iter = True
                        if self.verbose:
                            print(f"[HGS] t={t+1:02d}  ↑  fuzzy-score={F_best:.4f}  {params_new}")

            self.history.append(F_best)
            no_improve = 0 if improved_iter else (no_improve + 1)
            if no_improve >= self.patience:
                if self.verbose:
                    print(f"[HGS] early-stop @ t={t+1} (no improvement for {self.patience} iters)")
                break

        if self.verbose:
            print(f"[HGS] DONE   best fuzzy-score={self.best_score_:.4f}  "
                  f"params={self.best_params_}")
        return self.best_params_, self.best_score_, self.history


def hgs_search(search_space, fitness_fn,
               pop_size=8, max_iter=15, seed=42, verbose=True,
               early_stop_patience=4):
    """API simple : best_params, best_score, history = hgs_search(...)"""
    opt = HGSOptimizer(search_space, fitness_fn,
                       pop_size=pop_size, max_iter=max_iter,
                       seed=seed, verbose=verbose,
                       early_stop_patience=early_stop_patience)
    t0 = time.time()
    best_params, best_score, history = opt.optimize()
    print(f"[HGS] elapsed = {time.time()-t0:.1f}s")
    return best_params, best_score, history

In [ ]:
LR_SPACE = {
    "C":            [0.01, 0.05, 0.1, 0.3, 0.5, 1.0, 2.0],   # plafond abaissé
    "penalty":      ["l1", "l2", "elasticnet"],
    "solver":       ["liblinear", "saga", "lbfgs"],
    "class_weight": [None, "balanced"],
    "max_iter":     [500, 1000, 2000],
    "l1_ratio":     [0.1, 0.5, 0.9],
}

VALID_COMBOS = {
    ("l1", "liblinear"), ("l1", "saga"),
    ("l2", "liblinear"), ("l2", "saga"), ("l2", "lbfgs"),
    ("elasticnet", "saga"),
}

def build_estimator(params):
    p = dict(params)
    if (p["penalty"], p["solver"]) not in VALID_COMBOS:
        # combinaison invalide : on retombe sur (l2, lbfgs)
        p["penalty"], p["solver"] = "l2", "lbfgs"
    if p["penalty"] != "elasticnet":
        p.pop("l1_ratio", None)
    if p["solver"] == "liblinear":
        return LogisticRegression(**p, random_state=42, n_jobs=1)
    return LogisticRegression(**p, random_state=42, n_jobs=-1)


def canonical(params):
    # the model build_estimator() actually fits - used to merge duplicate grid points
    p = dict(params)
    if (p["penalty"], p["solver"]) not in VALID_COMBOS: p["penalty"], p["solver"] = "l2", "lbfgs"
    if p["penalty"] != "elasticnet": p["l1_ratio"] = None
    return tuple(sorted(p.items(), key=lambda kv: kv[0]))

DISTINCT = {}
for combo in itertools.product(*LR_SPACE.values()):
    p = dict(zip(LR_SPACE.keys(), combo)); DISTINCT.setdefault(canonical(p), p)
print("nominal grid:", int(np.prod([len(v) for v in LR_SPACE.values()])),
      "| distinct models:", len(DISTINCT))

In [ ]:
class Counted:
    # wraps a fitness so the number of DISTINCT models evaluated is known exactly
    def __init__(self, fn): self.fn, self.seen = fn, {}
    def __call__(self, params):
        k = canonical(params)
        if k not in self.seen: self.seen[k] = self.fn(params)
        return self.seen[k]
    @property
    def n(self): return len(self.seen)

def random_search(fitness, budget, seed):
    keys = list(DISTINCT.keys()); rng = random.Random(seed); rng.shuffle(keys)
    best_p, best_s = None, -1e9
    for k in keys[:budget]:
        s = fitness(DISTINCT[k])
        if s > best_s: best_p, best_s = DISTINCT[k], s
    return best_p, best_s

def grid_search(fitness):
    best_p, best_s = None, -1e9
    for i, p in enumerate(DISTINCT.values()):
        s = fitness(p)
        if s > best_s: best_p, best_s = p, s
        if (i+1) % 50 == 0: print("    grid", i+1, "/", len(DISTINCT), "best", round(best_s, 4))
    return best_p, best_s

def evaluate(y, proba, thr=0.5):
    yp = (proba >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yp, labels=[0,1]).ravel()
    return dict(
        accuracy=accuracy_score(y, yp),
        precision_pos=precision_score(y, yp, zero_division=0),
        recall_pos=recall_score(y, yp, zero_division=0),
        f1_pos=f1_score(y, yp, zero_division=0),
        f1_macro=f1_score(y, yp, average="macro", zero_division=0),
        precision_w=precision_score(y, yp, average="weighted", zero_division=0),
        recall_w=recall_score(y, yp, average="weighted", zero_division=0),
        f1_w=f1_score(y, yp, average="weighted", zero_division=0),
        roc_auc=roc_auc_score(y, proba), pr_auc=average_precision_score(y, proba),
        mcc=matthews_corrcoef(y, yp), bal_acc=balanced_accuracy_score(y, yp),
        TP=int(tp), FP=int(fp), FN=int(fn), TN=int(tn), n_test=int(len(y)))

## Autosave and resume

- Every model evaluation is cached to Drive the moment it finishes. HGS and random search are
  deterministic for a given seed, so after a disconnect a re-run replays the cached part instantly
  and continues from the exact model where it stopped - including midway through a grid.
- Every finished run is appended to this job's own CSV. Runs already in the CSV are skipped.
- T3 is split with stratification on the raw 4-way code, which reproduces Gulati et al.'s
  GPT-3.5 protocol exactly (1,260/420/840 train, 540/180/360 test). All other tasks keep the
  original stratification on the binary label, so their splits match the first run.

In [ ]:
import pickle
CACHE_DIR = os.path.join(OUT_DIR, "cache"); os.makedirs(CACHE_DIR, exist_ok=True)

def _atomic_pickle(path, obj):
    tmp = path + ".tmp"
    with open(tmp, "wb") as f: pickle.dump(obj, f)
    os.replace(tmp, path)

def _atomic_csv(df, path):
    tmp = path + ".tmp"; df.to_csv(tmp, index=False); os.replace(tmp, path)

class PersistentCounted:
    # fitness wrapper: saves each evaluation to Drive; n = distinct models queried by THIS search
    def __init__(self, fn, cache_path):
        self.fn, self.path = fn, cache_path
        self.cache = pickle.load(open(cache_path, "rb")) if os.path.exists(cache_path) else {}
        self.queried = set()
        if self.cache: print("    resuming from %d cached evaluations" % len(self.cache))
    def __call__(self, params):
        k = canonical(params); self.queried.add(k)
        if k not in self.cache:
            self.cache[k] = self.fn(params); _atomic_pickle(self.path, self.cache)
        return self.cache[k]
    @property
    def n(self): return len(self.queried)

def prepare(task, seed, sec=None, tag_correction=True):
    idx, y = task_index(task, seed)
    sec = SEC_NAME if sec is None else sec
    X = np.hstack([text_features(sec[idx], tag_correction), NUM_X[idx]]).astype(np.float32)
    strat = RAW_CODE[idx] if task == "T3" else y
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.30, random_state=seed, stratify=strat)
    sc = StandardScaler().fit(Xtr)                              # training split only
    return sc.transform(Xtr), sc.transform(Xte), ytr, yte

def run_saved(job, task, seed, strategy, variant="main", budget=None, sec=None, tag_correction=True):
    out = os.path.join(OUT_DIR, "results_%s.csv" % job)
    done = pd.read_csv(out) if os.path.exists(out) else pd.DataFrame()
    if len(done):
        hit = done[(done.task == task) & (done.seed == seed) &
                   (done.strategy == strategy) & (done.variant == variant)]
        if len(hit):
            print("%-4s seed %-4d %-7s %-18s already done - skipped" % (task, seed, strategy, variant))
            return hit.iloc[0].to_dict()
    Xtr, Xte, ytr, yte = prepare(task, seed, sec, tag_correction)
    cache = os.path.join(CACHE_DIR, "%s__%s_s%d_%s.pkl" % (job, task, seed, variant))
    fit = PersistentCounted(make_cv_fitness(build_estimator, Xtr, ytr, n_splits=5, seed=seed,
                                            orness=0.30, gap_penalty=0.50, use_proba=True), cache)
    t0 = time.time()
    if strategy == "HGS":
        bp, bs, _ = hgs_search(LR_SPACE, fit, seed=seed, verbose=False, **HGS_KW)
    elif strategy == "Random":
        bp, bs = random_search(fit, int(budget), seed)
    else:
        bp, bs = grid_search(fit)
    m = build_estimator(bp).fit(Xtr, ytr)
    r = evaluate(yte, m.predict_proba(Xte)[:, 1])
    r.update(job=job, task=task, seed=seed, strategy=strategy, variant=variant, cv_fitness=bs,
             n_models=fit.n, seconds=round(time.time() - t0, 1), params=json.dumps(bp),
             params_fitted=json.dumps(dict(canonical(bp))))
    _atomic_csv(pd.concat([done, pd.DataFrame([r])], ignore_index=True), out)
    print("%-4s seed %-4d %-7s %-18s models=%-3d F1+=%.4f acc=%.4f  (saved to %s)"
          % (task, seed, strategy, variant, r["n_models"], r["f1_pos"], r["accuracy"], os.path.basename(out)))
    return r

## Embedding variants (as in notebooks 07 and 09)

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression

def _unit(x):
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-12)

def _r2(E, direction):
    # Ren et al. R2: remove the component along `direction`, then normalise to unit length
    d = _unit(direction)
    return _unit(E - (E * d).sum(-1, keepdims=True) * d)

def _coherence(E):
    En = _unit(E)
    c = np.stack([(En[:, 0] * En[:, 1]).sum(1), (En[:, 0] * En[:, 2]).sum(1),
                  (En[:, 1] * En[:, 2]).sum(1)], 1)
    return np.c_[c, c.mean(1), c.min(1), c.max(1), c.std(1)]

def _l1(E):
    return np.stack([np.abs(E[:, 0] - E[:, 1]).sum(1), np.abs(E[:, 0] - E[:, 2]).sum(1),
                     np.abs(E[:, 1] - E[:, 2]).sum(1)], 1)

def embed_variant(v, Etr, Ete):
    # Etr, Ete: (n, 3, 768) raw section embeddings. Statistics come from Etr only.
    mu   = Etr.reshape(-1, Etr.shape[-1]).mean(0)          # global training mean
    mu_s = Etr.mean(0)                                     # per-section training means (3, 768)
    T    = TAG_E[None]                                     # tag embeddings (1, 3, 768)
    def f(E):
        if v == "V0": return (E - T).mean(1)
        if v == "V1": return E.mean(1)
        if v == "V2": return _unit(E - mu).mean(1)
        if v == "V3": return _r2(E, mu).mean(1)
        if v == "V4": return _r2(E, mu_s[None]).mean(1)
        if v == "V5": return _r2(E, T).mean(1)
        if v == "V7": return np.c_[(E - T).mean(1), _coherence(E)]
        if v == "V8":
            R = _r2(E, mu_s[None]); return np.c_[R.mean(1), _coherence(R)]
        if v == "V9": return np.c_[(E - T).mean(1), _l1(E)]
        raise ValueError(v)
    if v == "V6":
        A = Etr.reshape(-1, Etr.shape[-1]) - mu
        lam, U = np.linalg.eigh(np.cov(A, rowvar=False))
        lam = np.clip(lam, 0, None); eps = 1e-2 * lam.mean()   # shrinkage for the small eigenvalues
        W = U @ np.diag(1.0 / np.sqrt(lam + eps)) @ U.T
        return ((Etr - mu) @ W).mean(1), ((Ete - mu) @ W).mean(1)
    return f(Etr), f(Ete)

VARIANT_IDS = ["V0", "V1", "V2", "V3", "V4", "V5", "V6", "V7", "V8", "V9"]
C_GRID = [0.1, 0.5, 1.0]

def fit_screen(Xtr, ytr, seed):
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
    best_c, best_f = C_GRID[0], -1
    for c in C_GRID:
        fs = []
        for a, b in skf.split(Xtr, ytr):
            m = LogisticRegression(penalty="l1", solver="liblinear", C=c, class_weight="balanced",
                                   max_iter=2000, random_state=seed).fit(Xtr[a], ytr[a])
            fs.append(f1_score(ytr[b], m.predict(Xtr[b]), zero_division=0))
        if np.mean(fs) > best_f: best_c, best_f = c, np.mean(fs)
    m = LogisticRegression(penalty="l1", solver="liblinear", C=best_c, class_weight="balanced",
                           max_iter=2000, random_state=seed).fit(Xtr, ytr)
    return m, best_c, best_f

EMB_OUT = os.path.join(OUT_DIR, "results_embeddings.csv")

def run_variant(v, task, seed):
    done = pd.read_csv(EMB_OUT) if os.path.exists(EMB_OUT) else pd.DataFrame()
    if len(done) and ((done.variant == v) & (done.task == task) & (done.seed == seed)).any():
        return None
    idx, y = task_index(task, seed)
    strat = RAW_CODE[idx] if task == "T3" else y
    tr, te = train_test_split(np.arange(len(idx)), test_size=0.30, random_state=seed, stratify=strat)
    Etr, Ete = SEC_NAME[idx[tr]], SEC_NAME[idx[te]]
    Ztr, Zte = embed_variant(v, Etr, Ete)
    Xtr = np.hstack([Ztr, NUM_X[idx[tr]]]); Xte = np.hstack([Zte, NUM_X[idx[te]]])
    sc = StandardScaler().fit(Xtr); Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)
    t0 = time.time()
    m, c, cvf = fit_screen(Xtr, y[tr], seed)
    r = evaluate(y[te], m.predict_proba(Xte)[:, 1])
    r.update(variant=v, task=task, seed=seed, C=c, cv_f1=cvf, n_features=Xtr.shape[1],
             nonzero_coef=int((m.coef_ != 0).sum()), seconds=round(time.time() - t0, 1))
    _atomic_csv(pd.concat([done, pd.DataFrame([r])], ignore_index=True), EMB_OUT)
    print("%-4s %-4s seed %-4d C=%-4s F1+=%.4f AUC=%.4f MCC=%.4f" %
          (v, task, seed, c, r["f1_pos"], r["roc_auc"], r["mcc"]))
    return r

## Statistics and helpers

In [ ]:
# ---- statistics shared by notebooks 10-14 ----------------------------------------------------
from scipy import stats as sst

def _midrank(x):
    J = np.argsort(x, kind="mergesort"); Z = x[J]; N = len(x); T = np.zeros(N); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    out = np.empty(N); out[J] = T
    return out

def delong_test(y, s1, s2):
    """DeLong, DeLong and Clarke-Pearson (1988) test for two correlated ROC areas,
    computed with the fast algorithm of Sun and Xu (2014). Returns auc1, auc2, z, two-sided p."""
    y = np.asarray(y).astype(int); pos, neg = y == 1, y == 0; m, n = int(pos.sum()), int(neg.sum())
    aucs, V10, V01 = [], [], []
    for s in (np.asarray(s1, float), np.asarray(s2, float)):
        X, Y = s[pos], s[neg]
        tx, ty, tz = _midrank(X), _midrank(Y), _midrank(np.r_[X, Y])
        aucs.append((tz[:m].sum() - m * (m + 1) / 2) / (m * n))
        V10.append((tz[:m] - tx) / n); V01.append(1.0 - (tz[m:] - ty) / m)
    S10, S01 = np.cov(np.vstack(V10)), np.cov(np.vstack(V01))
    var = (S10[0, 0] + S10[1, 1] - 2 * S10[0, 1]) / m + (S01[0, 0] + S01[1, 1] - 2 * S01[0, 1]) / n
    if var <= 0:
        return aucs[0], aucs[1], 0.0, 1.0
    z = (aucs[0] - aucs[1]) / np.sqrt(var)
    return aucs[0], aucs[1], float(z), float(2 * sst.norm.sf(abs(z)))

def mcnemar_exact(y, pred1, pred2):
    """Exact McNemar test on the discordant pairs. Returns (only 1 right, only 2 right, two-sided p)."""
    y, pred1, pred2 = map(np.asarray, (y, pred1, pred2))
    n12 = int(((pred1 == y) & (pred2 != y)).sum()); n21 = int(((pred1 != y) & (pred2 == y)).sum())
    p = sst.binomtest(min(n12, n21), n12 + n21, 0.5).pvalue if n12 + n21 else 1.0
    return n12, n21, float(p)

def f1_pos_fast(y, p):
    tp = np.sum((p == 1) & (y == 1)); fp = np.sum((p == 1) & (y == 0)); fn = np.sum((p == 0) & (y == 1))
    return 2 * tp / max(2 * tp + fp + fn, 1)

def acc_fast(y, p):
    return float(np.mean(y == p))

def macro_f1_fast(y, p, k=None):
    k = int(max(y.max(), p.max()) + 1) if k is None else k
    cm = np.zeros((k, k)); np.add.at(cm, (y, p), 1)
    tp = np.diag(cm); prec = tp / np.maximum(cm.sum(0), 1); rec = tp / np.maximum(cm.sum(1), 1)
    return float(np.mean(2 * prec * rec / np.maximum(prec + rec, 1e-12)))

def _strat_resample(y, rng):
    return np.concatenate([rng.choice(np.where(y == c)[0], np.sum(y == c)) for c in np.unique(y)])

def bootstrap_ci(y, pred, metric, B=5000, seed=0):
    """Stratified bootstrap of the test set: point value, 95% percentile interval, standard error."""
    y, pred = np.asarray(y), np.asarray(pred); rng = np.random.default_rng(seed)
    v = np.array([metric(y[i], pred[i]) for i in (_strat_resample(y, rng) for _ in range(B))])
    return metric(y, pred), np.percentile(v, [2.5, 97.5]), float(v.std(ddof=1)), v

def paired_bootstrap(y, pred_a, pred_b, metric, B=5000, seed=0):
    """Difference metric(a) - metric(b) on the same test profiles, with a paired stratified bootstrap."""
    y, a, b = map(np.asarray, (y, pred_a, pred_b)); rng = np.random.default_rng(seed)
    d = np.array([metric(y[i], a[i]) - metric(y[i], b[i]) for i in (_strat_resample(y, rng) for _ in range(B))])
    return metric(y, a) - metric(y, b), np.percentile(d, [2.5, 97.5]), float(d.std(ddof=1))

def corrected_resampled_t(diffs, n_train, n_test, alternative="two-sided"):
    """Nadeau and Bengio (2003) corrected resampled t-test for J repeated random splits:
    the variance of the mean difference is inflated by (1/J + n_test/n_train) because training
    sets overlap between splits. Returns mean difference, t, p."""
    d = np.asarray(diffs, float); J = len(d)
    if J < 2 or np.allclose(d, d[0]):
        return float(d.mean()) if J else np.nan, np.nan, np.nan
    t = d.mean() / np.sqrt((1.0 / J + n_test / n_train) * d.var(ddof=1))
    p = sst.t.sf(t, J - 1) if alternative == "greater" else 2 * sst.t.sf(abs(t), J - 1)
    return float(d.mean()), float(t), float(p)

def wilcoxon_p(diffs, alternative="two-sided"):
    d = np.asarray(diffs, float)
    if np.all(d == 0) or len(d) < 2:
        return np.nan
    try:
        return float(sst.wilcoxon(d, alternative=alternative, zero_method="wilcox").pvalue)
    except ValueError:
        return np.nan

def sign_test_p(diffs, alternative="two-sided"):
    d = np.asarray(diffs, float); k, n = int((d > 0).sum()), int((d != 0).sum())
    return float(sst.binomtest(k, n, 0.5, alternative=alternative).pvalue) if n else np.nan

def seed_mean_ci(values, B=20000, seed=0):
    """Percentile interval for a mean over seeds, by resampling seeds (wide when there are few seeds)."""
    v = np.asarray(values, float); rng = np.random.default_rng(seed)
    return np.percentile(rng.choice(v, (B, len(v))).mean(1), [2.5, 97.5])

def min_detectable_difference(se_diff, alpha=0.05, power=0.80):
    """Smallest true difference a two-sided test at level alpha detects with the given power."""
    return (sst.norm.ppf(1 - alpha / 2) + sst.norm.ppf(power)) * se_diff

def fmt_p(p):
    return "n/a" if p is None or (isinstance(p, float) and np.isnan(p)) else ("<0.001" if p < 0.001 else "%.3f" % p)

In [ ]:
from sklearn.base import clone
PRED_DIR = os.path.join(OUT_DIR, "predictions"); os.makedirs(PRED_DIR, exist_ok=True)

def split_task(task, seed):
    """The exact partition of notebooks 00-09: stratified 70/30 (T3 stratified on the raw code)."""
    idx, y = task_index(task, seed)
    strat = RAW_CODE[idx] if task == "T3" else y
    tr, te = train_test_split(np.arange(len(idx)), test_size=0.30, random_state=seed, stratify=strat)
    return idx, y, tr, te

def features(variant, idx, tr, te, sec=None):
    """Embedding variant + 15 counts, standardised with training-split statistics only."""
    sec = SEC_NAME if sec is None else sec
    Ztr, Zte = embed_variant(variant, sec[idx[tr]], sec[idx[te]])
    Xtr = np.hstack([Ztr, NUM_X[idx[tr]]]); Xte = np.hstack([Zte, NUM_X[idx[te]]])
    sc = StandardScaler().fit(Xtr)
    return sc.transform(Xtr), sc.transform(Xte)

def save_pred(name, **arrays):
    np.savez_compressed(os.path.join(PRED_DIR, name + ".npz"), **arrays)

def load_pred(name):
    p = os.path.join(PRED_DIR, name + ".npz")
    return dict(np.load(p)) if os.path.exists(p) else None

def done_row(path, **key):
    if not os.path.exists(path): return None
    d = pd.read_csv(path)
    for k, v in key.items():
        if k not in d.columns: return None
        d = d[d[k] == v]
    return d.iloc[0].to_dict() if len(d) else None

def append_row(path, row):
    d = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()
    _atomic_csv(pd.concat([d, pd.DataFrame([row])], ignore_index=True), path)

## Settings

In [ ]:
# ---------------- switches ----------------
SENS_TASKS  = ["T1", "T1b", "T2", "T3"]        # one task per session if you run sessions in parallel
SENS_SEEDS  = [42, 7, 123]
ORNESS_GRID = [0.10, 0.20, 0.30, 0.50, 1.00]   # target orness; 0.30 is the paper's, 1.00 gives equal weights
LAMBDA_GRID = [0.0, 0.25, 0.50, 1.00, 2.00]    # gap penalty; 0.50 is the paper's
PAPER       = (0.30, 0.50)
RUN_OPTIMIZERS = True                          # random search, PSO and GA at the budget of HGS
N_THREADS   = 2                                # the six fits of one evaluation run in threads; 1 = sequential
HGS_KW = dict(pop_size=12, max_iter=18, early_stop_patience=4)

from joblib import Parallel, delayed
from sklearn.model_selection import StratifiedKFold
_R_ALL_PATH = os.path.join(OUT_DIR, "results_all.csv")
R_ALL = pd.read_csv(_R_ALL_PATH) if os.path.exists(_R_ALL_PATH) else None
if R_ALL is None: print("results_all.csv not found: the consistency check with the paper's runs is skipped")

def measured_orness(n, target):
    # weights are applied to the criteria sorted in ascending order, so index 0 is the minimum
    w = owa_weights(n, target); return float(np.dot(w, np.arange(n)) / (n - 1))
print("target orness -> weights on (worst, middle, best) criterion | measured orness for 3 criteria and 5 folds")
for o in ORNESS_GRID:
    print("  %.2f -> %s | %.4f  %.4f" % (o, np.round(owa_weights(3, o), 4), measured_orness(3, o), measured_orness(5, o)))

## Raw-criteria store and objective

In [ ]:
# ---- raw-criteria store: one fitted model serves every (orness, penalty) pair and every optimizer ----
def _criteria(m, Xb, yb):
    pred = m.predict(Xb)
    out = [f1_score(yb, pred, zero_division=0), recall_score(yb, pred, zero_division=0)]
    try:
        out.append(roc_auc_score(yb, m.predict_proba(Xb)[:, 1]))
    except Exception:
        pass
    return out

def _fit_fold(tmpl, Xa, ya, Xb, yb):
    try:
        return _criteria(clone(tmpl).fit(Xa, ya), Xb, yb)
    except Exception:
        return None

def _fit_full(tmpl, X, y, Xte):
    try:
        m = clone(tmpl).fit(X, y)
        return _criteria(m, X, y), m.predict_proba(Xte)[:, 1].astype(np.float32)
    except Exception:
        return None, None

class RawStore:
    # fits each distinct model once per task and split, and keeps what every objective needs
    def __init__(self, task, seed):
        self.task, self.seed = task, seed
        self.idx, self.y, self.tr, self.te = split_task(task, seed)
        self.Xtr, self.Xte = features("V0", self.idx, self.tr, self.te)
        self.ytr, self.yte = self.y[self.tr], self.y[self.te]
        self.folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
                          .split(np.zeros(len(self.ytr)), self.ytr))
        self.path = os.path.join(CACHE_DIR, "sens__raw_%s_s%d.pkl" % (task, seed))
        self.data = pickle.load(open(self.path, "rb")) if os.path.exists(self.path) else {}
        self.new, self.new_seconds = 0, 0.0
        if self.data: print("    %s seed %d: %d models already fitted" % (task, seed, len(self.data)))
    def get(self, params):
        k = canonical(params)
        if k in self.data: return self.data[k]
        t0 = time.time()
        try:
            tmpl = build_estimator(params)
        except Exception:
            tmpl = None
        if tmpl is None:
            e = dict(folds=None, train=None, test_proba=None)
        else:
            jobs = [delayed(_fit_fold)(tmpl, self.Xtr[a], self.ytr[a], self.Xtr[b], self.ytr[b]) for a, b in self.folds]
            jobs.append(delayed(_fit_full)(tmpl, self.Xtr, self.ytr, self.Xte))
            res = Parallel(n_jobs=N_THREADS, prefer="threads")(jobs) if N_THREADS > 1 else [f(*a, **kw) for f, a, kw in jobs]
            e = dict(folds=res[:5], train=res[5][0], test_proba=res[5][1])
        e["seconds"] = round(time.time() - t0, 1)
        self.data[k] = e; _atomic_pickle(self.path, self.data)
        self.new += 1; self.new_seconds += e["seconds"]
        return e

def fitness_from_raw(e, orness, lam):
    # identical to make_cv_fitness(): OWA over criteria, OWA over folds, minus lam x the train-CV gap
    if e["folds"] is None: return 0.0
    fs = [fuzzy_owa(c, orness) if c is not None else 0.0 for c in e["folds"]]
    owa_cv, mean_cv = fuzzy_owa(fs, orness), float(np.mean(fs))
    train = fuzzy_owa(e["train"], orness) if e["train"] is not None else mean_cv
    return owa_cv - lam * max(0.0, train - mean_cv)

class BudgetExhausted(Exception):
    pass

class Objective:
    # fitness seen by one search: counts its distinct models, enforces its budget, remembers its best
    def __init__(self, store, orness, lam, budget=None):
        self.store, self.orness, self.lam, self.budget = store, orness, lam, budget
        self.queried, self.best_p, self.best_s = {}, None, -np.inf
    def __call__(self, params):
        k = canonical(params)
        if k not in self.queried:
            if self.budget is not None and len(self.queried) >= self.budget: raise BudgetExhausted()
            self.queried[k] = fitness_from_raw(self.store.get(params), self.orness, self.lam)
            if self.queried[k] > self.best_s: self.best_p, self.best_s = dict(params), self.queried[k]
        return self.queried[k]
    @property
    def n(self): return len(self.queried)

## PSO and genetic algorithm

In [ ]:
# ---- competing optimizers: same grid, same decoding, same population size as HGS, same budget ----
KEYS = list(LR_SPACE.keys()); DIMS = [len(LR_SPACE[k]) for k in KEYS]; D = len(KEYS); POP = HGS_KW["pop_size"]

def decode(x):
    return {k: LR_SPACE[k][int(min(DIMS[i] - 1, max(0, math.floor(x[i] * DIMS[i]))))] for i, k in enumerate(KEYS)}

def pso_search(fit, seed, max_iter=300, stall=40):
    # global-best PSO with constriction coefficients (inertia 0.7298, c1 = c2 = 1.4962)
    rng = np.random.default_rng(seed)
    X = rng.random((POP, D)); V = (rng.random((POP, D)) - 0.5) * 0.2
    P, Pf, g, gf, idle = X.copy(), np.full(POP, -np.inf), X[0].copy(), -np.inf, 0
    try:
        for _ in range(max_iter):
            before = fit.n
            for i in range(POP):
                f = fit(decode(X[i]))
                if f > Pf[i]: Pf[i], P[i] = f, X[i].copy()
                if f > gf: gf, g = f, X[i].copy()
            idle = idle + 1 if fit.n == before else 0
            if idle >= stall: break
            r1, r2 = rng.random((POP, D)), rng.random((POP, D))
            V = 0.7298 * V + 1.4962 * r1 * (P - X) + 1.4962 * r2 * (g[None] - X)
            X = np.clip(X + V, 0.0, 1.0)
    except BudgetExhausted:
        pass
    return fit.best_p, fit.best_s

def ga_search(fit, seed, max_iter=300, stall=40, p_cross=0.9):
    # generational GA: tournament selection (3), uniform crossover, random-reset mutation (1/D per gene), one elite
    rng = np.random.default_rng(seed)
    X = rng.random((POP, D)); idle = 0
    try:
        F = np.array([fit(decode(x)) for x in X])
        for _ in range(max_iter):
            before = fit.n
            def pick():
                c = rng.choice(POP, 3, replace=False); return X[c[np.argmax(F[c])]]
            kids = []
            while len(kids) < POP - 1:
                a, b = pick().copy(), pick().copy()
                if rng.random() < p_cross:
                    m = rng.random(D) < 0.5; a[m], b[m] = b[m].copy(), a[m].copy()
                for c in (a, b):
                    mut = rng.random(D) < 1.0 / D; c[mut] = rng.random(int(mut.sum())); kids.append(c)
            kids = np.array(kids[:POP - 1]); elite = X[np.argmax(F)].copy(); Fe = F.max()
            Fk = np.array([fit(decode(x)) for x in kids])
            X, F = np.vstack([elite[None], kids]), np.r_[Fe, Fk]
            idle = idle + 1 if fit.n == before else 0
            if idle >= stall: break
    except BudgetExhausted:
        pass
    return fit.best_p, fit.best_s

## Run

In [ ]:
def sens_path(task): return os.path.join(OUT_DIR, "results_sensitivity_%s.csv" % task)
def run_id(strategy, o, lam): return "%s_o%.2f_l%.2f" % (strategy, o, lam)

def run_one(store, strategy, o, lam, budget=None):
    task, seed, rid = store.task, store.seed, run_id(strategy, o, lam)
    hit = done_row(sens_path(task), run=rid, seed=seed)
    if hit:
        print("%-4s seed %-4d %-18s already done - skipped" % (task, seed, rid)); return hit
    fit = Objective(store, o, lam, budget); new0, sec0 = store.new, store.new_seconds
    if strategy == "HGS":
        bp, bs, _ = hgs_search(LR_SPACE, fit, seed=seed, verbose=False, **HGS_KW)
    elif strategy == "Random":
        keys = list(DISTINCT.keys()); random.Random(seed).shuffle(keys)      # the order used in the paper's runs
        for k in keys[:budget]: fit(DISTINCT[k])
        bp, bs = fit.best_p, fit.best_s
    elif strategy == "PSO":
        bp, bs = pso_search(fit, seed)
    elif strategy == "GA":
        bp, bs = ga_search(fit, seed)
    else:
        raise ValueError(strategy)
    e = store.get(bp)
    if e["test_proba"] is None:
        print("WARNING: selected model could not be refitted - run skipped:", rid); return None
    proba = e["test_proba"].astype(float); r = evaluate(store.yte, proba)
    f1s = [c[0] if c is not None else 0.0 for c in e["folds"]]
    r.update(task=task, seed=seed, run=rid, strategy=strategy, orness=o, gap_penalty=lam, fitness=bs, n_models=fit.n,
             budget=budget if budget is not None else np.nan, cv_f1=float(np.mean(f1s)), cv_f1_sd=float(np.std(f1s)),
             train_f1=e["train"][0], optimism=float(np.mean(f1s)) - r["f1_pos"], overfit_gap=e["train"][0] - r["f1_pos"],
             n_train=int(len(store.ytr)), params_fitted=json.dumps(dict(canonical(bp))),
             new_models=store.new - new0, seconds_new=round(store.new_seconds - sec0, 1))
    save_pred("sens__%s_s%d_%s" % (task, seed, rid), y=store.yte, proba=proba, pred=(proba >= 0.5).astype(int),
              test_rows=store.idx[store.te])
    append_row(sens_path(task), r)
    print("%-4s seed %-4d %-18s models=%-3d (new %-3d, %5.0fs) test F1=%.4f | CV F1 %.4f" % (
        task, seed, rid, fit.n, r["new_models"], r["seconds_new"], r["f1_pos"], r["cv_f1"]))
    return r

for task in SENS_TASKS:
    for seed in SENS_SEEDS:
        store = RawStore(task, seed)
        base = run_one(store, "HGS", *PAPER)                      # the paper's setting first
        if RUN_OPTIMIZERS and base is not None:
            for s in ["Random", "PSO", "GA"]:
                run_one(store, s, *PAPER, budget=int(base["n_models"]))
        for o in ORNESS_GRID:
            if o != PAPER[0]: run_one(store, "HGS", o, PAPER[1])
        for lam in LAMBDA_GRID:
            if lam != PAPER[1]: run_one(store, "HGS", PAPER[0], lam)
print("all requested runs are done")

## Analysis

In [ ]:
import glob
from scipy import stats as sst
files = sorted(glob.glob(os.path.join(OUT_DIR, "results_sensitivity_*.csv")))
S = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
L = []
def say(*a):
    s = " ".join(str(x) for x in a); print(s); L.append(s)
TASKS_DONE = [t for t in ["T1", "T1b", "T2", "T3"] if t in set(S.task)]
BASE = S[S.run == run_id("HGS", *PAPER)].set_index(["task", "seed"])
say("runs: %d | tasks: %s | splits per task: %s" % (len(S), TASKS_DONE, S.groupby("task").seed.nunique().to_dict()))

say("\n=== 0. Consistency with the paper's stored runs (HGS, orness 0.30, penalty 0.50) ===")
if R_ALL is None:
    say("results_all.csv not available - not checked")
else:
    for (task, seed), b in BASE.iterrows():
        st = R_ALL[(R_ALL.variant == "main") & (R_ALL.strategy == "HGS") & (R_ALL.task == task) & (R_ALL.seed == seed)]
        if not len(st): say("%-4s seed %-4d no stored run" % (task, seed)); continue
        st = st.iloc[0]
        say("%-4s seed %-4d same model: %-5s | models %d vs %d | test F1 %.4f vs %.4f | fitness %.6f vs %.6f" % (
            task, seed, json.loads(st.params_fitted) == json.loads(b.params_fitted), b.n_models, st.n_models,
            b.f1_pos, st.f1_pos, b.fitness, st.cv_fitness))

say("\n=== 1. OWA weights for each target orness ===")
say("target  w(worst) w(middle) w(best)  measured orness (3 criteria)  (5 folds)")
for o in ORNESS_GRID:
    w = owa_weights(3, o); say("%.2f    %.4f   %.4f    %.4f   %.4f                        %.4f" % (
        o, w[0], w[1], w[2], measured_orness(3, o), measured_orness(5, o)))

def sweep(col, grid, fixed_col, fixed_val, title):
    say("\n=== %s ===" % title)
    say("Mean test F1 over splits (SD), by task:")
    hdr = "%-12s" % col + "".join("%-18s" % t for t in TASKS_DONE) + "all tasks"; say(hdr)
    means = {}
    for v in grid:
        A = S[(S.strategy == "HGS") & np.isclose(S[col], v) & np.isclose(S[fixed_col], fixed_val)]
        cells = []
        for t in TASKS_DONE:
            x = A[A.task == t].f1_pos.values
            cells.append("%.4f (%.4f)" % (x.mean(), x.std(ddof=1) if len(x) > 1 else 0) if len(x) else "-")
            means[(v, t)] = x.mean() if len(x) else np.nan
        say("%-12s" % ("%.2f" % v) + "".join("%-18s" % c for c in cells) + ("%.4f" % A.f1_pos.mean() if len(A) else "-"))
    say("Range of the mean F1 across the grid, in points: " + ", ".join(
        "%s %.2f" % (t, 100 * (np.nanmax([means[(v, t)] for v in grid]) - np.nanmin([means[(v, t)] for v in grid])))
        for t in TASKS_DONE))
    say("\nPaired with the paper's setting on the same task and split:")
    say("%-12s %5s  %-26s %-22s %-22s %-10s %s" % (col, "pairs", "test F1 diff, points (p)", "optimism diff (p)",
                                               "CV-F1 SD diff (p)", "same model", "better/tied/worse"))
    for v in grid:
        if np.isclose(v, PAPER[0] if col == "orness" else PAPER[1]): continue
        A = S[(S.strategy == "HGS") & np.isclose(S[col], v) & np.isclose(S[fixed_col], fixed_val)].set_index(["task", "seed"])
        common = A.index.intersection(BASE.index)
        if not len(common): continue
        out = []
        for c, k in [("f1_pos", 100), ("optimism", 100), ("cv_f1_sd", 100)]:
            d = (A.loc[common, c] - BASE.loc[common, c]).values * k
            out.append("%+.3f (%s)" % (d.mean(), fmt_p(wilcoxon_p(d))))
        d = (A.loc[common, "f1_pos"] - BASE.loc[common, "f1_pos"]).values
        same = np.mean(A.loc[common, "params_fitted"].values == BASE.loc[common, "params_fitted"].values)
        say("%-12s %5d  %-26s %-22s %-22s %-10s %d/%d/%d" % ("%.2f" % v, len(common), out[0], out[1], out[2],
            "%.0f%%" % (100 * same), (d > 1e-12).sum(), (np.abs(d) <= 1e-12).sum(), (d < -1e-12).sum()))

sweep("orness", ORNESS_GRID, "gap_penalty", PAPER[1], "2. Target orness (gap penalty 0.50)")
sweep("gap_penalty", LAMBDA_GRID, "orness", PAPER[0], "3. Gap penalty (target orness 0.30)")

say("\n=== 4. Does the fitness rank models like the test F1? (all fitted models of a task and split) ===")
say("Spearman correlation between fitness and test F1, mean over tasks and splits; and the regret of the model the")
say("fitness ranks first among the fitted ones (best test F1 among fitted models minus its test F1, in points).")
rows = []
for task in TASKS_DONE:
    for seed in sorted(S[S.task == task].seed.unique()):
        p = os.path.join(CACHE_DIR, "sens__raw_%s_s%d.pkl" % (task, seed))
        if not os.path.exists(p): continue
        raw = pickle.load(open(p, "rb")); idx, y, tr, te = split_task(task, seed); yte = y[te]
        ent = [e for e in raw.values() if e["folds"] is not None and e["test_proba"] is not None]
        tf1 = np.array([f1_pos_fast(yte, (e["test_proba"] >= 0.5).astype(int)) for e in ent])
        for o, lam in [(o, PAPER[1]) for o in ORNESS_GRID] + [(PAPER[0], l) for l in LAMBDA_GRID if l != PAPER[1]]:
            fit = np.array([fitness_from_raw(e, o, lam) for e in ent])
            rho = sst.spearmanr(fit, tf1).correlation if len(ent) > 2 else np.nan
            rows.append(dict(task=task, seed=seed, orness=o, gap_penalty=lam, n=len(ent), rho=rho,
                             regret=100 * (tf1.max() - tf1[int(np.argmax(fit))])))
RK = pd.DataFrame(rows)
if len(RK):
    say("fitted models per task and split: mean %.0f (min %d, max %d)" % (RK.n.mean(), RK.n.min(), RK.n.max()))
    say("%-7s %-8s %-10s %s" % ("orness", "penalty", "Spearman", "regret"))
    for (o, lam), g in RK.groupby(["orness", "gap_penalty"]):
        say("%-7.2f %-8.2f %-10.3f %.2f" % (o, lam, g.rho.mean(), g.regret.mean()))
    RK.to_csv(os.path.join(OUT_DIR, "sensitivity_ranking.csv"), index=False)

say("\n=== 5. Optimizers on the paper's objective, at the budget of HGS ===")
O = S[np.isclose(S.orness, PAPER[0]) & np.isclose(S.gap_penalty, PAPER[1])]
if set(O.strategy) - {"HGS"}:
    say("Mean over splits: test F1 | fitness reached | distinct models evaluated")
    for t in TASKS_DONE:
        say(t + ": " + " ; ".join("%s %.4f | %.4f | %.1f" % (s, g.f1_pos.mean(), g.fitness.mean(), g.n_models.mean())
                                   for s, g in O[O.task == t].groupby("strategy", sort=False)))
    say("\nHGS minus the other optimizer, test F1 in points (corrected resampled t-test over the splits of a task):")
    for s in ["Random", "PSO", "GA"]:
        A = O[O.strategy == s].set_index(["task", "seed"]); common = A.index.intersection(BASE.index)
        if not len(common): continue
        parts = []
        for t in TASKS_DONE:
            c = [k for k in common if k[0] == t]
            if len(c) < 2: continue
            d = 100 * (BASE.loc[c, "f1_pos"].values - A.loc[c, "f1_pos"].values)
            m, _, p = corrected_resampled_t(d, BASE.loc[c, "n_train"].iloc[0], BASE.loc[c, "n_test"].iloc[0])
            parts.append("%s %+.2f (p %s)" % (t, m, fmt_p(p)))
        d = 100 * (BASE.loc[common, "f1_pos"].values - A.loc[common, "f1_pos"].values)
        df = BASE.loc[common, "fitness"].values - A.loc[common, "fitness"].values
        same = np.mean(A.loc[common, "params_fitted"].values == BASE.loc[common, "params_fitted"].values)
        say("%-6s %s | all %d pairs: %+.3f (Wilcoxon p %s), HGS better/tied/worse %d/%d/%d | fitness diff %+.5f (p %s), "
            "HGS fitness higher/equal/lower %d/%d/%d | same model %.0f%%" % (
            s, " ; ".join(parts), len(common), d.mean(), fmt_p(wilcoxon_p(d)), (d > 1e-10).sum(), (np.abs(d) <= 1e-10).sum(),
            (d < -1e-10).sum(), df.mean(), fmt_p(wilcoxon_p(df)), (df > 1e-12).sum(), (np.abs(df) <= 1e-12).sum(),
            (df < -1e-12).sum(), 100 * same))
else:
    say("optimizer runs not found")

say("\n=== 6. Cost ===")
say("new models fitted: %d | fitting time: %.1f h | distinct models per search (mean): %s" % (
    S.new_models.sum(), S.seconds_new.sum() / 3600, S.groupby("strategy").n_models.mean().round(1).to_dict()))
open(os.path.join(OUT_DIR, "sensitivity_report.txt"), "w").write("\n".join(L))
print("\nsaved:", os.path.join(OUT_DIR, "sensitivity_report.txt"))

## Figure

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(7.2, 2.9), sharey=True)
marks = {"T1": "o", "T1b": "s", "T2": "^", "T3": "D"}
for a, col, grid, fc, fv, lab in [(ax[0], "orness", ORNESS_GRID, "gap_penalty", PAPER[1], "Target orness"),
                                  (ax[1], "gap_penalty", LAMBDA_GRID, "orness", PAPER[0], "Gap penalty")]:
    for t in TASKS_DONE:
        A = S[(S.strategy == "HGS") & (S.task == t) & np.isclose(S[fc], fv)]
        g = A.groupby(col).f1_pos.agg(["mean", "min", "max"]).reindex(grid)
        x = np.arange(len(grid))
        a.errorbar(x, g["mean"], yerr=[g["mean"] - g["min"], g["max"] - g["mean"]], marker=marks[t], ms=4, lw=1,
                   capsize=2, label=t)
    a.set_xticks(np.arange(len(grid))); a.set_xticklabels(["%.2f" % v for v in grid]); a.set_xlabel(lab)
    paper = grid.index(PAPER[0] if col == "orness" else PAPER[1]); a.axvline(paper, color="0.6", ls=":", lw=1)
    a.grid(axis="y", lw=0.3)
ax[0].set_ylabel("Test F1 (mean, min-max over splits)"); ax[1].legend(fontsize=7, ncol=2, frameon=False)
plt.tight_layout(); p = os.path.join(OUT_DIR, "fig_sensitivity.png"); plt.savefig(p, dpi=300); print("saved:", p)

## What to send back

From the results folder on Drive (`MyDrive/fake_profile_paper/linkedin_repro/`):

- `sensitivity_report.txt`
- `results_sensitivity_T1.csv`, `results_sensitivity_T1b.csv`, `results_sensitivity_T2.csv`, `results_sensitivity_T3.csv`
- `sensitivity_ranking.csv`
- `fig_sensitivity.png`